# Mini-projet — Segmentation d'images **v3 PRO**

Même tâche que la v1 (Penn-Fudan, personne vs fond), mais optimisée pour le **score**.

## Les 6 leviers, du plus au moins important

| # | Levier | Pourquoi ça gagne |
|---|---|---|
| 1 | **Encoder ResNet18 pré-entraîné** (ImageNet) | le réseau arrive en sachant déjà « voir » (contours, textures, formes) — sur 170 images c'est LE gros gain |
| 2 | **Résolution 256** (vs 128) | 4× plus de pixels → contours bien plus précis |
| 3 | **Augmentation riche** (crop aléatoire, color jitter) | multiplie artificiellement le dataset → moins de surapprentissage |
| 4 | **Normalisation ImageNet** | obligatoire avec un encoder pré-entraîné (il a appris sur des images normalisées comme ça) |
| 5 | **30 epochs + cosine LR + meilleur checkpoint** | on laisse converger, le LR descend en douceur, on garde le meilleur modèle |
| 6 | **AdamW + weight decay** | régularisation propre du fine-tuning |

**Référence v1 (from scratch, 128px, 5 epochs) : Dice test ≈ 0,64.**
Attendu ici : **nettement au-dessus** (l'ordre de 0,85+ est classique en transfer learning sur cette tâche — ton run le dira).

> Tourne en local (kernel `Python (GPU ROCm)`) comme sur Colab (T4).
> En local, la **1ʳᵉ epoch est plus lente** : MIOpen compile ses kernels pour chaque nouvelle forme, puis ça roule.

# 0 — Imports et configuration

In [ ]:
import os
import random
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparametres v3
IMG_SIZE = 256        # 4x plus de pixels que la v1 -> contours plus fins
BATCH_SIZE = 8        # reduire a 4 si memoire GPU insuffisante
EPOCHS = 30           # on laisse converger (le meilleur checkpoint est garde)
LEARNING_RATE = 3e-4  # AdamW
WEIGHT_DECAY = 1e-4   # regularisation
PRETRAINED = True     # encoder ResNet18 pre-entraine ImageNet <- LE levier n1

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# 1 — Dataset Penn-Fudan (téléchargement robuste)

`rglob` cherche `PNGImages/` et `PedMasks/` **où qu'ils soient** après décompression —
le zip officiel crée parfois un dossier imbriqué `PennFudanPed/PennFudanPed/`.

In [ ]:
DATA_ROOT = Path("../data/PennFudanPed")
ZIP_PATH = Path("../data/PennFudanPed.zip")
URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"

if not DATA_ROOT.exists():
    print("Telechargement du dataset (~50 Mo)...")
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(".")
    print("Telechargement termine.")
else:
    print("Dataset deja present.")

# Robuste aux deux structures possibles (avec ou sans dossier imbrique)
images_dir = next(DATA_ROOT.rglob("PNGImages"))
masks_dir = next(DATA_ROOT.rglob("PedMasks"))

image_paths = sorted(images_dir.glob("*.png"))
mask_paths = sorted(masks_dir.glob("*.png"))

assert len(image_paths) == len(mask_paths) > 0
print(f"\n{len(image_paths)} couples image/masque disponibles")

In [ ]:
pairs = list(zip(image_paths, mask_paths))
random.Random(SEED).shuffle(pairs)

n_total = len(pairs)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train + n_val]
test_pairs = pairs[n_train + n_val:]

print(f"Train      : {len(train_pairs)}")
print(f"Validation : {len(val_pairs)}")
print(f"Test       : {len(test_pairs)}")

assert set(train_pairs).isdisjoint(val_pairs)
assert set(train_pairs).isdisjoint(test_pairs)
assert set(val_pairs).isdisjoint(test_pairs)

# 2 — Dataset PyTorch avec augmentation renforcée

Ce qui change vs v1 :
- **Crop aléatoire à échelle variable** (70–100 % de l'image) — le modèle voit des personnes à des tailles différentes
- **Color jitter** (luminosité/contraste/saturation) — image seule, jamais le masque
- **Normalisation ImageNet** après `to_tensor` — obligatoire avec l'encoder pré-entraîné
- Le flip horizontal est conservé

Règle inchangée : toute transformation **géométrique** s'applique à l'image **ET** au masque
(mêmes paramètres), les transformations de **couleur** à l'image seule.

## 🧠 Le cours — normalisation ImageNet et augmentation

### Pourquoi ces nombres magiques ?

```python
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # moyenne des canaux R, G, B...
IMAGENET_STD  = [0.229, 0.224, 0.225]   # ...et leur ecart-type
```

Ce sont la **moyenne et l'écart-type des 1,2 million d'images d'ImageNet** — les données
sur lesquelles le ResNet18 a été pré-entraîné. Il a appris à voir des images normalisées
**comme ça** : si tu lui donnes du 0-1 brut, ses filtres reçoivent des valeurs décalées
et le transfer learning marche beaucoup moins bien. C'est le `StandardScaler` du cours,
avec le μ/σ **d'ImageNet** au lieu du tien — même logique que « fit sur le train,
transform partout » : on réutilise les statistiques du *pré-entraînement*.

Conséquence : `denormalize()` pour l'affichage (sinon les couleurs sont fausses à
l'écran), et la **même normalisation** obligatoire à l'inférence (Mission 7).

### L'augmentation v3, transformation par transformation

| Transformation | S'applique à | Pourquoi |
|---|---|---|
| `hflip` (p=0,5) | image **ET** masque | une personne miroir reste une personne |
| `RandomResizedCrop.get_params` (70-100 %) | image **ET** masque, **mêmes i,j,h,w** | le modèle voit des personnes à des échelles différentes |
| `ColorJitter` (p=0,8) | image **SEULE** | luminosité/contraste varient dans la vraie vie — mais ne changent pas OÙ est la personne |

**La règle absolue** : géométrique → les deux, avec les *mêmes* paramètres tirés une
seule fois. Photométrique (couleur) → l'image seule, jamais le masque.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

_color_jitter = transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05)


def denormalize(t):
    """Pour l'affichage : annule la normalisation ImageNet."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1)


class SegPersonDataset(Dataset):
    def __init__(self, pairs, size=256, augment=False):
        self.pairs = list(pairs)
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.augment:
            # --- geometrique : memes parametres pour image ET masque ---
            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                image, scale=(0.7, 1.0), ratio=(0.75, 1.33))
            image = TF.resized_crop(image, i, j, h, w, [self.size, self.size],
                                    interpolation=InterpolationMode.BILINEAR, antialias=True)
            mask = TF.resized_crop(mask, i, j, h, w, [self.size, self.size],
                                   interpolation=InterpolationMode.NEAREST)
            # --- photometrique : image SEULE ---
            if random.random() < 0.8:
                image = _color_jitter(image)
        else:
            image = TF.resize(image, [self.size, self.size],
                              interpolation=InterpolationMode.BILINEAR, antialias=True)
            mask = TF.resize(mask, [self.size, self.size],
                             interpolation=InterpolationMode.NEAREST)

        image_tensor = TF.to_tensor(image)
        image_tensor = TF.normalize(image_tensor, IMAGENET_MEAN, IMAGENET_STD)

        mask_np = (np.array(mask) > 0).astype(np.float32)
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0)
        return image_tensor, mask_tensor


# --- Verification rapide ---
_dbg = SegPersonDataset(train_pairs[:3], size=IMG_SIZE, augment=True)
_img, _msk = _dbg[0]
assert _img.shape == (3, IMG_SIZE, IMG_SIZE)
assert _msk.shape == (1, IMG_SIZE, IMG_SIZE)
assert set(torch.unique(_msk).tolist()).issubset({0.0, 1.0})
print(f"Dataset OK — image {tuple(_img.shape)}, masque {tuple(_msk.shape)}")

In [ ]:
train_dataset = SegPersonDataset(train_pairs, IMG_SIZE, augment=True)
val_dataset = SegPersonDataset(val_pairs, IMG_SIZE, augment=False)
test_dataset = SegPersonDataset(test_pairs, IMG_SIZE, augment=False)

# num_workers=0 : indispensable sous Windows/Jupyter (le multiprocessing y plante)
PIN = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=PIN)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=PIN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=PIN)

images_batch, masks_batch = next(iter(train_loader))
print(f"Batch images  : {images_batch.shape}")
print(f"Batch masques : {masks_batch.shape}")

In [ ]:
# Verification visuelle (denormalisee pour l'affichage)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    axes[0, i].imshow(denormalize(images_batch[i]).permute(1, 2, 0))
    axes[0, i].set_title(f"Image #{i} (augmentee)")
    axes[0, i].axis("off")
    axes[1, i].imshow(masks_batch[i, 0], cmap="gray")
    axes[1, i].set_title(f"Masque #{i}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

# 3 — Le modèle : U-Net à encoder **ResNet18 pré-entraîné**

**Le levier n°1.** Le `TinyUNet` de la v1 partait de poids aléatoires : il devait apprendre
à « voir » avec 119 images. Ici, l'encoder est un **ResNet18 entraîné sur ImageNet**
(1,2 million d'images) : il sait déjà détecter contours, textures et formes. On ne lui
apprend plus qu'à les **assembler en masque** — c'est le principe du *transfer learning*.

Structure : les 4 étages du ResNet servent d'encoder, un decoder U-Net classique remonte
avec les **skip connections** prises à chaque étage. ~20 M de paramètres (vs 118 K en v1) —
mais la majorité arrive déjà entraînée.

## 🧠 Le cours — le transfer learning, LE levier n°1

### L'idée en une phrase

Un réseau entraîné sur **1,2 million d'images** (ImageNet) a déjà appris ce que sont un
contour, une texture, une forme, un visage. On **réutilise** cette vision acquise et on
ne lui apprend plus que la fin : « assemble ce que tu vois en masque de personne ».

Avec 170 (ou 2667) images, apprendre à voir **depuis zéro** est impossible — c'était la
limite du TinyUNet. Réutiliser 11 M de poids déjà entraînés change la donne.

### L'anatomie du ResNet18 utilisé en encoder

| Étage | Résolution (entrée 256) | Canaux | Rôle dans le U-Net |
|---|---|---|---|
| `layer0` (conv 7×7) | 128 (/2) | 64 | skip0 — détail fin |
| `layer1` | 64 (/4) | 64 | skip1 |
| `layer2` | 32 (/8) | 128 | skip2 |
| `layer3` | 16 (/16) | 256 | skip3 |
| `layer4` | 8 (/32) | 512 | le bottleneck |

C'est **exactement** la structure encoder → bottleneck du TinyUNet, en plus profond
(5 étages au lieu de 2) et pré-entraîné. Le decoder, lui, reste un U-Net classique
(`ConvTranspose2d` + `torch.cat` + `DoubleConv`) — il part de poids aléatoires, c'est
lui qui apprend la tâche.

`weights=ResNet18_Weights.DEFAULT` télécharge les poids ImageNet (~45 Mo, une fois).
`weights=None` donnerait le même réseau **vierge** — utile pour mesurer ce que le
pré-entraînement apporte vraiment.

In [ ]:
class DoubleConv(nn.Module):
    """(Conv 3x3 -> BN -> ReLU) x2 — la brique standard du U-Net."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResNetUNet(nn.Module):
    """U-Net dont l'encoder est un ResNet18 (pre-entraine ImageNet si weights)."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
        base = torchvision.models.resnet18(weights=weights)

        # --- Encoder (etages du ResNet) ---
        self.layer0 = nn.Sequential(base.conv1, base.bn1, base.relu)  # /2,  64 ch
        self.maxpool = base.maxpool                                    # /4
        self.layer1 = base.layer1                                      # /4,  64 ch
        self.layer2 = base.layer2                                      # /8,  128 ch
        self.layer3 = base.layer3                                      # /16, 256 ch
        self.layer4 = base.layer4                                      # /32, 512 ch

        # --- Decoder (remontee + skip connections) ---
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512, 256)   # cat avec skip3 (256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)   # cat avec skip2 (128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)    # cat avec skip1 (64)
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)    # cat avec skip0 (64)
        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec0 = DoubleConv(32, 32)
        self.head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        skip0 = self.layer0(x)             # /2
        x = self.maxpool(skip0)            # /4
        skip1 = self.layer1(x)             # /4
        skip2 = self.layer2(skip1)         # /8
        skip3 = self.layer3(skip2)         # /16
        x = self.layer4(skip3)             # /32

        x = self.dec4(torch.cat([self.up4(x), skip3], dim=1))
        x = self.dec3(torch.cat([self.up3(x), skip2], dim=1))
        x = self.dec2(torch.cat([self.up2(x), skip1], dim=1))
        x = self.dec1(torch.cat([self.up1(x), skip0], dim=1))
        x = self.dec0(self.up0(x))
        return self.head(x)


model = ResNetUNet(pretrained=PRETRAINED).to(DEVICE)

dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
out = model(dummy)
assert out.shape == (2, 1, IMG_SIZE, IMG_SIZE), f"Sortie inattendue : {out.shape}"

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametres entrainables : {n_params:,}")
print(f"Input  : {tuple(dummy.shape)}")
print(f"Output : {tuple(out.shape)}")

# 4 — Loss et métriques (identiques à la v1)

BCE + Soft Dice moitié-moitié pour l'entraînement, Dice + IoU (après seuillage) pour
l'évaluation. Rien à changer ici : c'était déjà le bon choix.

In [ ]:
BCE = nn.BCEWithLogitsLoss()


def soft_dice_loss(logits, targets, eps=1e-6):
    probabilities = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probabilities * targets).sum(dim=dims)
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)
    dice = (2 * intersection + eps) / (denominator + eps)
    return 1 - dice.mean()


def combined_loss(logits, targets):
    return 0.5 * BCE(logits, targets) + 0.5 * soft_dice_loss(logits, targets)


@torch.no_grad()
def binary_metrics(logits, targets, threshold=0.5, eps=1e-6):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()
    dims = (1, 2, 3)
    intersection = (predictions * targets).sum(dim=dims)
    pred_area = predictions.sum(dim=dims)
    target_area = targets.sum(dim=dims)
    union = pred_area + target_area - intersection
    dice = (2 * intersection + eps) / (pred_area + target_area + eps)
    iou = (intersection + eps) / (union + eps)
    return dice, iou

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    all_dice = []
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = combined_loss(logits, masks)
        loss.backward()
        optimizer.step()
        dice, _ = binary_metrics(logits.detach(), masks)
        total_loss += loss.item() * images.size(0)
        all_dice.extend(dice.cpu().tolist())
    return {"loss": total_loss / len(loader.dataset), "dice": float(np.mean(all_dice))}


@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_dice = []
    all_iou = []
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        logits = model(images)
        loss = combined_loss(logits, masks)
        dice, iou = binary_metrics(logits, masks, threshold)
        total_loss += loss.item() * images.size(0)
        all_dice.extend(dice.cpu().tolist())
        all_iou.extend(iou.cpu().tolist())
    return {"loss": total_loss / len(loader.dataset),
            "dice": float(np.mean(all_dice)),
            "iou": float(np.mean(all_iou))}

# 5 — Entraînement : AdamW + cosine LR + meilleur checkpoint

- **AdamW** avec `weight_decay` : la variante d'Adam qui régularise proprement
- **CosineAnnealingLR** : le learning rate descend en douceur de `3e-4` vers ~0 —
  gros pas au début (apprendre vite), pas fins à la fin (peaufiner sans casser)
- Le **meilleur état** (Dice validation) est conservé et rechargé à la fin

## 🧠 Le cours — AdamW et le cosine schedule

### AdamW vs Adam

Le **weight decay** tire doucement tous les poids vers zéro à chaque pas — ça décourage
les poids énormes, donc le par-cœur (régularisation). Adam classique mélange maladroitement
ce decay avec son adaptation du pas ; **AdamW le découple** et l'applique proprement.
En fine-tuning, c'est le choix standard.

### CosineAnnealingLR — le lr qui suit un cosinus

```
lr
3e-4 ━╮
       ╰─╮          grandes enjambees au debut : apprendre vite
          ╰──╮
             ╰───╮  petits pas a la fin : peaufiner sans rien casser
~0 ─────────────╰━  epoch 30
```

Le lr descend de `3e-4` vers ~0 en suivant une demi-période de cosinus. Fixe (v1), le lr
de fin est trop grand : le modèle « rebondit » autour du minimum sans s'y poser.

⚠️ `scheduler.step()` s'appelle **une fois par epoch** (pas par batch), après la
validation. Et le **meilleur checkpoint** reste la ceinture de sécurité : même si les
dernières epochs surapprennent, on garde la meilleure photo des poids.

In [ ]:
model = ResNetUNet(pretrained=PRETRAINED).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "train_dice": [], "val_loss": [], "val_dice": []}
best_val_dice = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_stats = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_stats = evaluate(model, val_loader, DEVICE)
    scheduler.step()

    history["train_loss"].append(train_stats["loss"])
    history["train_dice"].append(train_stats["dice"])
    history["val_loss"].append(val_stats["loss"])
    history["val_dice"].append(val_stats["dice"])

    if val_stats["dice"] > best_val_dice:
        best_val_dice = val_stats["dice"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        marker = "  <- meilleur"
    else:
        marker = ""

    lr_now = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | lr={lr_now:.2e} | "
          f"train loss={train_stats['loss']:.3f} dice={train_stats['dice']:.3f} | "
          f"val loss={val_stats['loss']:.3f} dice={val_stats['dice']:.3f}{marker}")

assert best_state is not None
model.load_state_dict(best_state)
print(f"\nMeilleur Dice de validation : {best_val_dice:.4f}")

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_range, history["train_loss"], "o-", label="Train")
axes[0].plot(epochs_range, history["val_loss"], "s-", label="Validation")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_dice"], "o-", label="Train")
axes[1].plot(epochs_range, history["val_dice"], "s-", label="Validation")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
axes[1].set_title("Dice score"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

# 6 — Seuil optimal (sur la validation) puis évaluation finale (test)

In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
validation_scores = []

for threshold in thresholds:
    stats = evaluate(model, val_loader, DEVICE, threshold=threshold)
    validation_scores.append(stats["dice"])
    print(f"Seuil {threshold:.2f} -> Dice validation = {stats['dice']:.4f}")

best_index = int(np.argmax(validation_scores))
best_threshold = thresholds[best_index]
print(f"\nMeilleur seuil : {best_threshold:.2f}")

In [ ]:
test_stats = evaluate(model, test_loader, DEVICE, threshold=best_threshold)

print("=== Resultats finaux sur le test set (v3) ===")
print(f"Loss  : {test_stats['loss']:.4f}")
print(f"Dice  : {test_stats['dice']:.4f}")
print(f"IoU   : {test_stats['iou']:.4f}")
print()
print("Rappel v1 (TinyUNet from scratch, 128px, 5 epochs) : Dice ~0.64, IoU ~0.49")

# 7 — Visualisation des prédictions

In [ ]:
@torch.no_grad()
def show_predictions(model, loader, threshold, n=4):
    model.eval()
    images, masks = next(iter(loader))
    logits = model(images.to(DEVICE))
    probabilities = torch.sigmoid(logits).cpu()
    predictions = (probabilities >= threshold).float()

    n = min(n, len(images))
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        axes[i, 0].imshow(denormalize(images[i]).permute(1, 2, 0))
        axes[i, 1].imshow(masks[i, 0], cmap="gray")
        axes[i, 2].imshow(probabilities[i, 0], cmap="gray", vmin=0, vmax=1)
        axes[i, 3].imshow(predictions[i, 0], cmap="gray")
        for j in range(4):
            axes[i, j].axis("off")
    axes[0, 0].set_title("Image")
    axes[0, 1].set_title("Verite terrain")
    axes[0, 2].set_title("Probabilite")
    axes[0, 3].set_title("Prediction (seuillee)")
    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, best_threshold, n=4)

# 8 — Ta propre photo (optionnel)

Dépose une `.jpg`/`.png` avec une personne dans `photos_test/` puis exécute.
La cellule **ne plante pas** s'il n'y a pas de photo — elle te le dit et passe.

In [ ]:
PHOTOS_DIR = Path("../photos_test")
PHOTOS_DIR.mkdir(exist_ok=True)
photos = list(PHOTOS_DIR.glob("*.jpg")) + list(PHOTOS_DIR.glob("*.png"))

prediction = None
if not photos:
    print(f"Aucune image dans {PHOTOS_DIR} — depose une photo puis relance cette cellule.")
else:
    photo_path = photos[0]
    print(f"Image testee : {photo_path.name}")

    new_image = Image.open(photo_path).convert("RGB")
    resized = TF.resize(new_image, [IMG_SIZE, IMG_SIZE],
                        interpolation=InterpolationMode.BILINEAR, antialias=True)
    input_tensor = TF.to_tensor(resized)
    input_tensor = TF.normalize(input_tensor, IMAGENET_MEAN, IMAGENET_STD)
    input_tensor = input_tensor.unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)

    probability = torch.sigmoid(logits).squeeze().cpu().numpy()
    prediction = (probability >= best_threshold).astype(np.float32)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(new_image); axes[0].set_title("Photo originale"); axes[0].axis("off")
    axes[1].imshow(probability, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Probabilite predite"); axes[1].axis("off")
    axes[2].imshow(prediction, cmap="gray")
    axes[2].set_title(f"Masque (seuil {best_threshold:.2f})"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Bonus fond vert / fond flou (necessite opencv : pip install opencv-python)
if prediction is None:
    print("Pas de photo testee — rien a faire.")
else:
    try:
        import cv2
    except ImportError:
        cv2 = None
        print("opencv absent : pip install opencv-python pour le fond flou.")

    original_size = new_image.size
    prediction_pil = Image.fromarray((prediction * 255).astype(np.uint8))
    prediction_full = np.array(prediction_pil.resize(original_size, Image.NEAREST)) > 127
    image_np = np.array(new_image)

    result_green = image_np.copy()
    result_green[~prediction_full] = [0, 255, 0]

    panels = [(image_np, "Original"), (result_green, "Fond vert")]
    if cv2 is not None:
        image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
        blurred = cv2.GaussianBlur(image_bgr, (45, 45), 0)
        result_blur = image_bgr.copy()
        result_blur[~prediction_full] = blurred[~prediction_full]
        result_blur = cv2.cvtColor(result_blur, cv2.COLOR_BGR2RGB)
        panels.append((result_blur, "Fond flou (style Zoom)"))

    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
    if len(panels) == 1:
        axes = [axes]
    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img); ax.set_title(title); ax.axis("off")
    plt.tight_layout()
    plt.show()

# Bilan v1 vs v3

| | v1 | v3 |
|---|---|---|
| Modèle | TinyUNet from scratch (118 K params) | **ResNet18-UNet pré-entraîné** (~20 M params) |
| Résolution | 128 | **256** |
| Augmentation | flip | **flip + crop échelle + color jitter** |
| Normalisation | 0-1 | **ImageNet** |
| Optimiseur | Adam, lr fixe | **AdamW + cosine LR** |
| Epochs | 5 | **30** (meilleur checkpoint gardé) |
| Dice test | ~0,64 | *(ton run l'affiche ci-dessus)* |

## Pour aller encore plus loin (v4 ?)

- **Plus de données** : fusionner Penn-Fudan avec le dataset Kaggle Supervisely (2667 images) de la v2
- **Architecture** : DeepLabV3+ (`torchvision.models.segmentation`) ou `segmentation_models_pytorch`
- **TTA** (test-time augmentation) : prédire l'image + son miroir, moyenner les probabilités
- **Résolution 384/512** si la mémoire GPU le permet